In [ ]:
# El modelo llega entrenado; hay que comprobar que sigue siendo confiable al operarlo.

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

ACTIVITY_DIR = Path('..').resolve()
DATA_DIR = ACTIVITY_DIR / 'data'
SUBMISSION_DIR = ACTIVITY_DIR / 'submission'
MIN_ACCURACY = 0.70
MIN_BALANCED_ACCURACY = 0.70
MIN_AUC = 0.80


In [ ]:
# Se carga el artefacto para probar su operación, no para reentrenarlo.

with (ACTIVITY_DIR / 'estimator.pkl').open('rb') as file:
    model = pickle.load(file)

test_set = pd.read_csv(DATA_DIR / 'model_test_set.csv')
features = list(model.feature_names_in_)
X_test = test_set.loc[:, features]
y_test = test_set['target']

features, X_test.shape, sorted(model.classes_.tolist())


In [ ]:
# El modelo debe entregar predicciones y probabilidades utilizables por quien lo consume.

def test_prediction_interface(model, inputs):
    predictions = model.predict(inputs)
    probabilities = model.predict_proba(inputs)

    assert len(predictions) == len(inputs)
    assert probabilities.shape == (len(inputs), len(model.classes_))
    assert set(predictions).issubset(set(model.classes_))
    assert np.all((probabilities >= 0) & (probabilities <= 1))
    assert np.allclose(probabilities.sum(axis=1), 1.0)

    return {'rows_checked': len(inputs), 'status': 'passed'}


In [ ]:
# Los casos de referencia protegen expectativas simples que no deberían cambiar sin revisión.

def test_known_behavior(model, features):
    reference_cases = pd.DataFrame(
        [
            {'texture_mean': 13.06, 'compactness_mean': 0.03774},
            {'texture_mean': 24.91, 'compactness_mean': 0.26650},
        ]
    ).loc[:, features]
    probabilities = model.predict_proba(reference_cases)[:, 1]

    assert probabilities[1] > probabilities[0]

    return {
        'low_reference_probability': float(probabilities[0]),
        'high_reference_probability': float(probabilities[1]),
        'status': 'passed',
    }


In [ ]:
# El desempeño mínimo evita habilitar un artefacto que ya no cumple lo acordado.

def test_holdout_performance(model, inputs, target):
    predictions = model.predict(inputs)
    probabilities = model.predict_proba(inputs)[:, 1]
    accuracy = accuracy_score(target, predictions)
    balanced_accuracy = balanced_accuracy_score(target, predictions)
    auc = roc_auc_score(target, probabilities)

    assert accuracy >= MIN_ACCURACY
    assert balanced_accuracy >= MIN_BALANCED_ACCURACY
    assert auc >= MIN_AUC

    return {
        'accuracy': float(accuracy),
        'balanced_accuracy': float(balanced_accuracy),
        'auc': float(auc),
        'status': 'passed',
    }


In [ ]:
# La misma entrada debe producir el mismo resultado para que una decisión sea auditable.

def test_reproducible_scoring(model, inputs):
    first_score = model.predict_proba(inputs)
    second_score = model.predict_proba(inputs)

    assert np.allclose(first_score, second_score)

    return {'rows_checked': len(inputs), 'status': 'passed'}


In [ ]:
# Las pruebas reúnen evidencia para decidir si el artefacto sigue habilitado.

results = {
    'prediction_interface': test_prediction_interface(model, X_test),
    'known_behavior': test_known_behavior(model, features),
    'holdout_performance': test_holdout_performance(model, X_test, y_test),
    'reproducible_scoring': test_reproducible_scoring(model, X_test),
}

pd.DataFrame(results).T


In [ ]:
# El reporte conserva la evidencia fuera de esta sesión.

report = {
    'model_file': 'estimator.pkl',
    'features': features,
    'holdout_rows': len(X_test),
    'thresholds': {
        'min_accuracy': MIN_ACCURACY,
        'min_balanced_accuracy': MIN_BALANCED_ACCURACY,
        'min_auc': MIN_AUC,
    },
    'tests': results,
}

SUBMISSION_DIR.mkdir(exist_ok=True)
report_path = SUBMISSION_DIR / 'model_test_report.json'
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

print(f'Reporte generado: {report_path.name}')
